<a href="https://colab.research.google.com/github/KarunyaVarma/Karunya-Varma/blob/main/DNN_GTPP_Regression1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this line once in Colab if keras-tuner is not installed:
!pip install -q keras-tuner

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -------------------------------------------------
# 1. Input Data & Reproducibility
# -------------------------------------------------

tf.keras.utils.set_random_seed(42)

data = pd.read_csv('/content/GTPP-DATASET.csv')

# 8 input features
X = data[[
    'AT',
    'AP',
    'DF',
    'CH',
    'GP',
    'CPR',
    'CPD',
    'TTXM'
]].values

# Target output
y = data['EP'].values


# -------------------------------------------------
# 2. Train-Test Split (80:20 Ratio)
# -------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# -------------------------------------------------
# 3. MinMax Scaling
# -------------------------------------------------

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# -------------------------------------------------
# 4. Define the 2-Layer Search Space (Explicit Sequential)
# -------------------------------------------------

def build_model(hp):
    # Define hyperparameter choices
    hp_units_1 = hp.Int('units_1', min_value=64, max_value=256, step=32)
    hp_units_2 = hp.Int('units_2', min_value=32, max_value=128, step=16)
    hp_activation = hp.Choice('activation', values=['relu', 'elu', 'swish'])
    hp_lr = hp.Float('learning_rate', min_value=5e-4, max_value=5e-3, sampling='log')

    # Explicit 2-Layer Sequential Model
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(8,)),
        tf.keras.layers.Dense(hp_units_1, activation=hp_activation),
        tf.keras.layers.Dense(hp_units_2, activation=hp_activation),
        tf.keras.layers.Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_lr),
        loss='mse',
        metrics=['mae']
    )

    return model


# -------------------------------------------------
# 5. Initialize & Run Bayesian Optimization Tuner
# -------------------------------------------------

tuner = kt.BayesianOptimization(
    hypermodel=build_model,
    objective='val_loss',
    max_trials=30,
    executions_per_trial=1,
    seed=42,
    directory='gtpp_tuner_dir',
    project_name='phase3_2layer_tuning',
    overwrite=True
)

early_stop_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

print("Starting Phase 3: Automated Bayesian Optimization (30 Trials)...\n")

tuner.search(
    X_train_scaled,
    y_train,
    epochs=200,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop_callback],
    verbose=1
)


# -------------------------------------------------
# 6. Extract Best Hyperparameters & Evaluate Best Model
# -------------------------------------------------

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.get_best_models(num_models=1)[0]

y_pred = best_model.predict(X_test_scaled, verbose=0).flatten()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n==========================================================================")
print("               PHASE 3: BEST AUTOMATED 2-LAYER MODEL                      ")
print("==========================================================================")
print(f"Best Layer 1 Neurons : {best_hps.get('units_1')}")
print(f"Best Layer 2 Neurons : {best_hps.get('units_2')}")
print(f"Best Activation      : {best_hps.get('activation')}")
print(f"Best Learning Rate   : {best_hps.get('learning_rate'):.6f}")
print(f"Total Parameters     : {best_model.count_params()}")
print("--------------------------------------------------------------------------")
print(f"Test MAE             : {mae:.6f}")
print(f"Test MSE             : {mse:.6f}")
print(f"Test RMSE            : {rmse:.6f}")
print(f"Test R2 Score        : {r2:.6f}")
print("==========================================================================")

Trial 30 Complete [00h 06m 08s]
val_loss: 1.5935182571411133

Best val_loss So Far: 1.5408926010131836
Total elapsed time: 03h 10m 01s


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



               PHASE 3: BEST AUTOMATED 2-LAYER MODEL                      
Best Layer 1 Neurons : 224
Best Layer 2 Neurons : 128
Best Activation      : swish
Best Learning Rate   : 0.000500
Total Parameters     : 30945
--------------------------------------------------------------------------
Test MAE             : 1.010376
Test MSE             : 1.721465
Test RMSE            : 1.312046
Test R2 Score        : 0.956139
